In [18]:
import pandas as pd

df_n = pd.read_csv("data/PubChem_compound_natural_dye.csv")
df_s = pd.read_csv("data/PubChem_compound_reactive_dye.csv")

In [19]:
son_identicas = df_s.columns.equals(df_n.columns)
print(f"¿Son exactamente iguales?: {son_identicas}") # Devuelve True

¿Son exactamente iguales?: True


In [20]:
df_n.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 515 entries, 0 to 514
Data columns (total 38 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Compound_CID                        515 non-null    int64  
 1   Name                                515 non-null    object 
 2   Synonyms                            514 non-null    object 
 3   Molecular_Formula                   515 non-null    object 
 4   InChI                               515 non-null    object 
 5   SMILES                              515 non-null    object 
 6   InChIKey                            515 non-null    object 
 7   IUPAC_Name                          513 non-null    object 
 8   MeSH_Headings                       250 non-null    object 
 9   Annotation_Content                  514 non-null    object 
 10  Linked_BioAssays                    308 non-null    object 
 11  Data_Source                         515 non-n

In [21]:
import pandas as pd
from rdkit import Chem
from rdkit import RDLogger 

# Silenciamos los errores de C++ en la consola
lg = RDLogger.logger()
lg.setLevel(RDLogger.CRITICAL)

# Precompilamos el patrón SMARTS fuera de la función para que sea ultrarrápido
PATRON_HETEROAROMATICO = Chem.MolFromSmarts("[a;!#6]")

def es_cromoforo(smiles):
    """
    Evalúa si una molécula tiene al menos un anillo heteroaromático
    usando coincidencia de subestructuras (A prueba de versiones RDKit).
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        
        if mol is None:
            return False
            
        # HasSubstructMatch devolverá True si encuentra el patrón
        return mol.HasSubstructMatch(PATRON_HETEROAROMATICO)
        
    except Exception as e:
        print(f"Error procesando {smiles}: {e}")
        return False

In [22]:
df_n['Es_Cromoforo'] = df_n['SMILES'].apply(es_cromoforo)

In [23]:
# 1. Filtramos las filas donde es_cromoforo es True
# 2. Usamos .drop() para eliminar la columna
# 3. .copy() asegura que el nuevo DF sea independiente
df_fn = df_n[df_n['Es_Cromoforo'] == True].drop(columns=['Es_Cromoforo']).copy()

# Verificamos el resultado
print(f"Moléculas restantes: {len(df_fn)}")
print("Columnas actuales:", df_fn.columns.tolist())

Moléculas restantes: 209
Columnas actuales: ['Compound_CID', 'Name', 'Synonyms', 'Molecular_Formula', 'InChI', 'SMILES', 'InChIKey', 'IUPAC_Name', 'MeSH_Headings', 'Annotation_Content', 'Linked_BioAssays', 'Data_Source', 'Data_Source_Category', 'Tagged_by_PubChem', 'Molecular_Weight', 'Polar_Area', 'Complexity', 'XLogP', 'Heavy_Atom_Count', 'H-Bond_Donor_Count', 'H-Bond_Acceptor_Count', 'Rotatable_Bond_Count', 'Exact_Mass', 'Monoisotopic_Mass', 'Charge', 'Covalent_Unit_Count', 'Isotopic_Atom_Count', 'Total_Atom_Stereo_Count', 'Defined_Atom_Stereo_Count', 'Undefined_Atom_Stereo_Count', 'Total_Bond_Stereo_Count', 'Defined_Bond_Stereo_Count', 'Undefined_Bond_Stereo_Count', 'Linked_PubChem_Literature_Count', 'Linked_PubChem_Patent_Count', 'Linked_PubChem_Patent_Family_Count', 'Annotation_Type_Count', 'Create_Date']


In [24]:
df_fn['es_natural'] = True
df_s['es_natural'] = False

In [25]:
df_final = pd.concat([df_fn, df_s], ignore_index=True)

In [26]:
df_final.head()

,Compound_CID,Name,Synonyms,Molecular_Formula,InChI,SMILES,InChIKey,IUPAC_Name,MeSH_Headings,Annotation_Content,...,Undefined_Atom_Stereo_Count,Total_Bond_Stereo_Count,Defined_Bond_Stereo_Count,Undefined_Bond_Stereo_Count,Linked_PubChem_Literature_Count,Linked_PubChem_Patent_Count,Linked_PubChem_Patent_Family_Count,Annotation_Type_Count,Create_Date,es_natural
0,11048,Eosin,Acid red 87|Eosin|17372-87-1|Eosin yellowish|E...,C20H6Br4Na2O5,InChI=1S/C20H8Br4O5.2Na/c21-11-5-9-13(7-3-1-2-...,C1=CC=C(C(=C1)C2=C3C=C(C(=O)C(=C3OC4=C(C(=C(C=...,SEACYXSIPDVVMV-UHFFFAOYSA-L,"disodium;2-(2,4,5,7-tetrabromo-3-oxido-6-oxoxa...",Eosine Yellowish-(YS),Biological Test Results|Interactions and Pathw...,...,0,0,0,0,103941,27902,12126,14,20050801,True
1,15410449,DND-99 dye,DND-99 dye|CHEBI:52117|RefChem:1083929|Lysotra...,C20H24BF2N5O,InChI=1S/C20H24BF2N5O/c1-26(2)13-12-25-20(29)1...,[B-]1(N2C(=CC=C2CCC(=O)NCCN(C)C)C=C3[N+]1=C(C=...,DYYUXMKNXUZBMO-UHFFFAOYSA-N,"3-[2,2-difluoro-12-(1H-pyrrol-2-yl)-3-aza-1-az...",NaN,Classification|Literature|Patents,...,0,0,0,0,1,682,371,3,20070209,True
2,54609158,Cyanine dye 863,Cyanine dye 863|Cyanine dye no. 863|NSC43683|N...,C24H24ClN3,InChI=1S/C24H24N3.ClH/c1-4-27-21(15-14-19-10-8...,CCN1/C(=C\C2=[N+](C(=NC(=C2)C)C3=CC=CC=C3)C)/C...,IGQBMVUAVTYSTO-UHFFFAOYSA-M,"(2Z)-2-[(3,6-dimethyl-2-phenylpyrimidin-3-ium-...",NaN,Biological Test Results|Literature|Patents|Bio...,...,0,1,1,0,1,0,0,4,20111219,True
3,71528224,"MK-2 Dye, 95%","MK-2 Dye|MK-2 Dye, 95%|SCHEMBL14964454|(E)-2-c...",C58H70N2O2S4,InChI=1S/C58H70N2O2S4/c1-6-11-15-19-25-40-34-5...,CCCCCCC1=C(SC(=C1)C2=C(C=C(S2)C3=C(C=C(S3)C4=C...,FOELOZKDLHJOHT-XLDZHHEVSA-N,(E)-2-cyano-3-[5-[5-[5-[5-(9-ethylcarbazol-3-y...,NaN,Literature|Patents,...,0,1,1,0,0,12,6,2,20130611,True
4,71433733,D149;Indoline dye D149,D149 Dye|786643-20-7|D149;Indoline dye D149|D-...,C42H35N3O4S3,InChI=1S/C42H35N3O4S3/c1-2-43-40(49)38(52-42(4...,CCN1C(=O)C(=C2N(C(=O)C(=CC3=CC4=C(C=C3)N(C5C4C...,OZFUEQNYOBIXTB-UHFFFAOYSA-N,"2-[5-[[4-[4-(2,2-diphenylethenyl)phenyl]-2,3,3...",NaN,Literature|Patents,...,2,2,0,2,2,57,30,2,20130522,True


In [27]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 223 entries, 0 to 222
Data columns (total 39 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Compound_CID                        223 non-null    int64  
 1   Name                                223 non-null    object 
 2   Synonyms                            223 non-null    object 
 3   Molecular_Formula                   223 non-null    object 
 4   InChI                               223 non-null    object 
 5   SMILES                              223 non-null    object 
 6   InChIKey                            223 non-null    object 
 7   IUPAC_Name                          220 non-null    object 
 8   MeSH_Headings                       97 non-null     object 
 9   Annotation_Content                  221 non-null    object 
 10  Linked_BioAssays                    110 non-null    object 
 11  Data_Source                         223 non-n

In [28]:
df_final.columns

Index(['Compound_CID', 'Name', 'Synonyms', 'Molecular_Formula', 'InChI',
       'SMILES', 'InChIKey', 'IUPAC_Name', 'MeSH_Headings',
       'Annotation_Content', 'Linked_BioAssays', 'Data_Source',
       'Data_Source_Category', 'Tagged_by_PubChem', 'Molecular_Weight',
       'Polar_Area', 'Complexity', 'XLogP', 'Heavy_Atom_Count',
       'H-Bond_Donor_Count', 'H-Bond_Acceptor_Count', 'Rotatable_Bond_Count',
       'Exact_Mass', 'Monoisotopic_Mass', 'Charge', 'Covalent_Unit_Count',
       'Isotopic_Atom_Count', 'Total_Atom_Stereo_Count',
       'Defined_Atom_Stereo_Count', 'Undefined_Atom_Stereo_Count',
       'Total_Bond_Stereo_Count', 'Defined_Bond_Stereo_Count',
       'Undefined_Bond_Stereo_Count', 'Linked_PubChem_Literature_Count',
       'Linked_PubChem_Patent_Count', 'Linked_PubChem_Patent_Family_Count',
       'Annotation_Type_Count', 'Create_Date', 'es_natural'],
      dtype='object')

In [29]:
int_colmumns = [5, 14, 15, 16, 17, 18, 19, 20, 24, 25, 38]

final = df_final.copy()
final = final.iloc[:, int_colmumns]

In [30]:
final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 223 entries, 0 to 222
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   SMILES                 223 non-null    object 
 1   Molecular_Weight       223 non-null    float64
 2   Polar_Area             223 non-null    float64
 3   Complexity             223 non-null    int64  
 4   XLogP                  98 non-null     float64
 5   Heavy_Atom_Count       223 non-null    int64  
 6   H-Bond_Donor_Count     223 non-null    int64  
 7   H-Bond_Acceptor_Count  223 non-null    int64  
 8   Charge                 223 non-null    int64  
 9   Covalent_Unit_Count    223 non-null    int64  
 10  es_natural             223 non-null    bool   
dtypes: bool(1), float64(3), int64(6), object(1)
memory usage: 17.8+ KB


In [31]:
final.to_csv('data/dataset.csv', index=False)